In [0]:
# COMMAND ----------
# 04 - TIME SERIES FORECASTING

import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

df = spark.table("fraud.risk_scored_full").toPandas()
df["date"] = pd.to_datetime(df["date"])

ts = (
    df.groupby("date")["amount"]
    .sum()
    .sort_index()
    .reset_index()
    .rename(columns={"amount": "cash_outflow"})
)

model = ARIMA(ts["cash_outflow"], order=(5, 1, 0))
fit = model.fit()
ts["forecast"] = fit.predict()

spark.createDataFrame(ts).write.mode("overwrite").saveAsTable("fraud.daily_cash_forecast")
